In [ ]:
%pip install scanpy anndata scikit-misc transformers numba rpy2 gdown


In [ ]:
import os
import scipy
import anndata
import sklearn
import torch
import random
import copy
import numpy as np
import scanpy as sc
import pandas as pd
from typing import Optional
import scipy.sparse as sp
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch.nn.modules.module import Module
from torch.backends import cudnn
from scipy.sparse import coo_matrix
from scipy.sparse import issparse
from sklearn.neighbors import NearestNeighbors
from sklearn.neighbors import kneighbors_graph
from sklearn.utils import sparsefuncs
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    v_measure_score,
    silhouette_score
)
from sklearn.preprocessing import LabelEncoder
import numba
from tqdm import tqdm
from transformers import AutoModel

def init_weights(*params):
    for param in params:
        torch.nn.init.xavier_uniform_(param)

def fix_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

def construct_neighbor_graph(adata_omics1, adata_omics2, datatype='SPOTS', n_neighbors=3):
    if datatype in ['Stereo-CITE-seq', 'Spatial-epigenome-transcriptome']:
        n_neighbors = 6
    cell_position_omics1 = adata_omics1.obsm['spatial']
    adata_omics1.uns['adj_spatial'] = construct_graph_by_coordinate(cell_position_omics1, n_neighbors=n_neighbors)
    cell_position_omics2 = adata_omics2.obsm['spatial']
    adata_omics2.uns['adj_spatial'] = construct_graph_by_coordinate(cell_position_omics2, n_neighbors=n_neighbors)
    feature_graph_omics1, feature_graph_omics2 = construct_graph_by_feature(adata_omics1, adata_omics2)
    adata_omics1.obsm['adj_feature'], adata_omics2.obsm['adj_feature'] = feature_graph_omics1, feature_graph_omics2
    data = {'adata_omics1': adata_omics1, 'adata_omics2': adata_omics2}
    return data

def pca(adata, use_reps=None, n_comps=10):
    from sklearn.decomposition import PCA
    from scipy.sparse import csc_matrix, csr_matrix
    pca_model = PCA(n_components=n_comps)
    if use_reps is not None:
        feat_pca = pca_model.fit_transform(adata.obsm[use_reps])
    else:
        if isinstance(adata.X, csc_matrix) or isinstance(adata.X, csr_matrix):
            feat_pca = pca_model.fit_transform(adata.X.toarray())
        else:
            feat_pca = pca_model.fit_transform(adata.X)
    return feat_pca

def clr_normalize_each_cell(adata, inplace=True):
    def seurat_clr(x):
        s = np.sum(np.log1p(x[x > 0]))
        exp = np.exp(s / len(x)) if len(x) > 0 else 1.0
        return np.log1p(x / exp)
    if not inplace:
        adata = adata.copy()
    adata.X = np.apply_along_axis(
        seurat_clr, 1, (adata.X.toarray() if scipy.sparse.issparse(adata.X) else np.array(adata.X))
    )
    return adata

def construct_graph_by_feature(adata_omics1, adata_omics2, k=20, mode="connectivity", metric="correlation", include_self=False):
    feature_graph_omics1 = kneighbors_graph(adata_omics1.obsm['feat'], k, mode=mode, metric=metric, include_self=include_self)
    feature_graph_omics2 = kneighbors_graph(adata_omics2.obsm['feat'], k, mode=mode, metric=metric, include_self=include_self)
    return feature_graph_omics1, feature_graph_omics2

def construct_graph_by_coordinate(cell_position, n_neighbors=3):
    nbrs = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(cell_position)
    _, indices = nbrs.kneighbors(cell_position)
    x = indices[:, 0].repeat(n_neighbors)
    y = indices[:, 1:].flatten()
    adj = pd.DataFrame({'x': x, 'y': y, 'value': np.ones(x.size)})
    return adj

def transform_adjacent_matrix(adjacent):
    n_spot = adjacent['x'].max() + 1
    adj = coo_matrix((adjacent['value'], (adjacent['x'], adjacent['y'])), shape=(n_spot, n_spot))
    return adj

def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse.FloatTensor(indices, values, shape)

def preprocess_graph(adj):
    adj = sp.coo_matrix(adj)
    adj_ = adj + sp.eye(adj.shape[0])
    rowsum = np.array(adj_.sum(1))
    degree_mat_inv_sqrt = sp.diags(np.power(rowsum, -0.5).flatten())
    adj_normalized = adj_.dot(degree_mat_inv_sqrt).transpose().dot(degree_mat_inv_sqrt).tocoo()
    return sparse_mx_to_torch_sparse_tensor(adj_normalized)

def adjacent_matrix_preprocessing(adata_omics1, adata_omics2, adj_emb):
    def _process_adj(adj):
        adj = adj.toarray() + adj.toarray().T
        adj = np.where(adj > 1, 1, adj)
        return preprocess_graph(adj)
    adj_spatial_omics1 = _process_adj(transform_adjacent_matrix(adata_omics1.uns['adj_spatial']))
    adj_spatial_omics2 = _process_adj(transform_adjacent_matrix(adata_omics2.uns['adj_spatial']))
    adj_emb = _process_adj(adj_emb)
    def _process_feature_adj(adj):
        adj = adj + adj.T
        return preprocess_graph(np.where(adj > 1, 1, adj))
    adj_feature_omics1 = _process_feature_adj(torch.FloatTensor(adata_omics1.obsm['adj_feature'].copy().toarray()))
    adj_feature_omics2 = _process_feature_adj(torch.FloatTensor(adata_omics2.obsm['adj_feature'].copy().toarray()))
    return {'adj_spatial_omics1': adj_spatial_omics1, 'adj_spatial_omics2': adj_spatial_omics2,
            'adj_feature_omics1': adj_feature_omics1, 'adj_feature_omics2': adj_feature_omics2, 'adj_emb': adj_emb}

def lsi(adata: anndata.AnnData, n_components: int = 20, use_highly_variable: Optional[bool] = None, **kwargs):
    if use_highly_variable is None:
        use_highly_variable = "highly_variable" in adata.var
    adata_use = adata[:, adata.var["highly_variable"]] if use_highly_variable else adata
    X = tfidf(adata_use.X)
    X_norm = sklearn.preprocessing.Normalizer(norm="l1").fit_transform(X)
    X_norm = np.log1p(X_norm * 1e4)
    X_lsi = sklearn.utils.extmath.randomized_svd(X_norm, n_components, **kwargs)[0]
    X_lsi -= X_lsi.mean(axis=1, keepdims=True)
    X_lsi /= X_lsi.std(axis=1, ddof=1, keepdims=True)
    adata.obsm["X_lsi"] = X_lsi[:, 1:]

def tfidf(X):
    idf = X.shape[0] / X.sum(axis=0)
    if scipy.sparse.issparse(X):
        tf = X.multiply(1 / X.sum(axis=1))
        return tf.multiply(idf)
    else:
        tf = X / X.sum(axis=1, keepdims=True)
        return tf * idf

def sf_normalize(X):
    X = X.copy()
    counts = np.array(X.sum(axis=1))
    counts += counts == 0.
    scaling_factor = 10000. / counts
    if issparse(X):
        sparsefuncs.inplace_row_scale(X, scaling_factor)
    else:
        np.multiply(X, scaling_factor.reshape((-1, 1)), out=X)
    return X

@numba.jit(nopython=True, nogil=True)
def _sub_tokenize_data(x, max_seq_len=-1, aux_tokens=30):
    scores_final = np.empty((x.shape[0], max_seq_len if max_seq_len > 0 else x.shape[1]))
    for i, cell in enumerate(x):
        nonzero_mask = np.nonzero(cell)[0]
        sorted_indices = nonzero_mask[np.argsort(-cell[nonzero_mask])][:max_seq_len]
        sorted_indices = sorted_indices + aux_tokens
        scores = np.zeros(max_seq_len if max_seq_len > 0 else cell.shape[0], dtype=np.int32)
        scores[:len(sorted_indices)] = sorted_indices.astype(np.int32)
        scores_final[i, :] = scores
    return scores_final

def tokenize_for_nicheformer(X_raw, max_seq_len=1500, aux_tokens=30):
    X = X_raw.copy()
    if issparse(X): X = X.toarray()
    X = np.nan_to_num(X)
    X = sf_normalize(X)
    median_counts_per_gene = np.median(X, axis=0)
    median_counts_per_gene += median_counts_per_gene == 0
    X = X / median_counts_per_gene.reshape((1, -1))
    return _sub_tokenize_data(X, max_seq_len, aux_tokens).astype(np.int32)

def extract_nicheformer_embeddings(model, tokens_np, batch_size=16, layer=-1):
    model.eval()
    all_embs = []
    n_cells = tokens_np.shape[0]
    padding_token = 1
    with torch.no_grad():
        for start in range(0, n_cells, batch_size):
            end = min(start + batch_size, n_cells)
            batch_tokens = torch.LongTensor(tokens_np[start:end]).to(next(model.parameters()).device)
            batch_tokens = torch.where(batch_tokens == 0, torch.tensor(padding_token, device=batch_tokens.device), batch_tokens)
            attention_mask = (batch_tokens == padding_token).bool()
            try:
                outputs = model(input_ids=batch_tokens, attention_mask=attention_mask)
                hidden = outputs.last_hidden_state if hasattr(outputs, 'last_hidden_state') else outputs[0]
            except TypeError:
                token_embedding = model.embeddings(batch_tokens)
                hidden = token_embedding
            hidden = hidden[:, 3:, :].mean(dim=1)
            all_embs.append(hidden.cpu())
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    return torch.cat(all_embs, dim=0)

class DeepEncoder(Module):
    def __init__(self, in_feat, out_feat, dropout=0.0, act=F.relu):
        super().__init__()
        self.dropout = dropout
        self.act = act
        self.hidden_dim = out_feat * 2
        self.weights = torch.nn.ParameterList([
            Parameter(torch.FloatTensor(in_feat, self.hidden_dim)),
            Parameter(torch.FloatTensor(self.hidden_dim, self.hidden_dim)),
            Parameter(torch.FloatTensor(self.hidden_dim, out_feat))
        ])
        init_weights(*self.weights)

    def forward(self, feat, adj):
        x = F.dropout(self.act(torch.spmm(adj, torch.mm(feat, self.weights[0]))), self.dropout, training=self.training)
        x = F.dropout(self.act(torch.spmm(adj, torch.mm(x, self.weights[1]))), self.dropout, training=self.training)
        return torch.spmm(adj, torch.mm(x, self.weights[2]))

class CellEmbedding(Module):
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.weight = Parameter(torch.FloatTensor(in_feat, out_feat))
        init_weights(self.weight)
    def forward(self, feat, adj):
        return torch.spmm(adj, torch.mm(feat, self.weight))

class AttentionLayer(Module):
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.w_omega = Parameter(torch.FloatTensor(in_feat, out_feat))
        self.u_omega = Parameter(torch.FloatTensor(out_feat, 1))
        init_weights(self.w_omega, self.u_omega)
    def forward(self, *embeddings):
        emb_stack = torch.cat([torch.unsqueeze(emb, dim=1) for emb in embeddings], dim=1)
        v = torch.tanh(torch.matmul(emb_stack, self.w_omega))
        alpha = F.softmax(torch.matmul(v, self.u_omega).squeeze(-1) + 1e-6, dim=1)
        return torch.matmul(emb_stack.transpose(1, 2), alpha.unsqueeze(-1)).squeeze(-1), alpha

class EncodingNetwork(Module):
    def __init__(self, dim_in_omics1, dim_out_omics1, dim_in_omics2, dim_out_omics2):
        super().__init__()
        self.encoder_embedding = CellEmbedding(512, 64)
        self.decoder_embedding = CellEmbedding(64, 512)
        self.encoder_omics1 = DeepEncoder(dim_in_omics1, dim_out_omics1)
        self.decoder_omics1 = DeepEncoder(dim_out_omics1, dim_in_omics1)
        self.encoder_omics2 = DeepEncoder(dim_in_omics2, dim_out_omics2)
        self.decoder_omics2 = DeepEncoder(dim_out_omics2, dim_in_omics2)
        self.atten_feature1 = AttentionLayer(dim_out_omics1, dim_out_omics1)
        self.atten_feature2 = AttentionLayer(dim_out_omics1, dim_out_omics1)
        self.atten_feature = AttentionLayer(dim_out_omics1, dim_out_omics1)
        self.atten_omics2 = AttentionLayer(dim_out_omics2, dim_out_omics2)
        self.atten_cross = AttentionLayer(dim_out_omics1, dim_out_omics2)

    def forward(self, f_omics1, f_omics2, adj_spa1, adj_fea1, adj_spa2, adj_fea2, cell_emb, adj_emb):
        emb_spa = self.encoder_embedding(cell_emb, adj_spa1)
        emb_fea = self.encoder_embedding(cell_emb, adj_emb)
        emb_latent_spa1 = self.encoder_omics1(f_omics1, adj_spa1)
        emb_latent_spa2 = self.encoder_omics2(f_omics2, adj_spa2)
        emb_latent_fea1 = self.encoder_omics1(f_omics1, adj_fea1)
        emb_latent_fea2 = self.encoder_omics2(f_omics2, adj_fea2)
        emb_att1, alpha_att1 = self.atten_feature1(emb_spa, emb_latent_spa1)
        emb_att2, alpha_att2 = self.atten_feature2(emb_fea, emb_latent_fea1)
        emb_latent_omics1, alpha_att_omics1 = self.atten_feature(emb_att1, emb_att2)
        emb_latent_omics2, alpha_omics2 = self.atten_omics2(emb_latent_spa2, emb_latent_fea2)
        emb_latent_combined, alpha = self.atten_cross(emb_latent_omics1, emb_latent_omics2)
        emb_recon1 = self.decoder_omics1(emb_latent_combined, adj_spa1)
        emb_recon2 = self.decoder_omics2(emb_latent_combined, adj_spa2)
        emb_recon_spa = self.decoder_embedding(emb_spa, adj_spa1)
        emb_recon_fea = self.decoder_embedding(emb_fea, adj_emb)
        emb_cross1 = self.encoder_omics2(self.decoder_omics2(emb_latent_omics1, adj_spa2), adj_spa2)
        emb_cross2 = self.encoder_omics1(self.decoder_omics1(emb_latent_omics2, adj_spa1), adj_spa1)
        return {'emb_latent_omics1': emb_latent_omics1, 'emb_latent_omics2': emb_latent_omics2,
                'emb_latent_combined': emb_latent_combined, 'emb_recon_omics1': emb_recon1, 'emb_recon_omics2': emb_recon2,
                'emb_cross1': emb_cross1, 'emb_cross2': emb_cross2,
                'alpha_att1': alpha_att1, 'alpha_att2': alpha_att2, 'alpha_omics1': alpha_att_omics1,
                'alpha_omics2': alpha_omics2, 'alpha': alpha, 'emb_recon_spa': emb_recon_spa, 'emb_recon_fea': emb_recon_fea}

def add_gaussian_noise(matrix, mean=0.0, std=0.001):
    return matrix + torch.normal(mean=mean, std=std, size=matrix.size()).to(matrix.device)

class Train_spaLLM:
    def __init__(self, data, embedding, datatype='10x', device=torch.device('cpu'),
                 epochval=None, random_seed=2024, learning_rate=0.0001, weight_decay=0.0, epochs=600,
                 dim_input=3000, dim_output=64, weight_factors=None, attention_type='local'):
        self.device = device
        self.data = data.copy()
        self.embedding = torch.FloatTensor(embedding).to(device) if isinstance(embedding, np.ndarray) else embedding.to(device)
        self.datatype = datatype
        self.random_seed = random_seed
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.dim_output = dim_output
        self.weight_factors = weight_factors or [5, 5, 1, 10, 10, 10]
        self._init_adj_and_features()
        self.loss_history = []
        if self.datatype == 'SPOTS': self.epochs, self.weight_factors = 600, [1, 5, 1, 1, 5, 5]
        elif self.datatype in ['10x', 'Stereo-CITE-seq']: self.epochs, self.weight_factors = 200, [5, 5, 1, 10, 10, 10]
        elif self.datatype == 'Spatial-epigenome-transcriptome': self.epochs, self.weight_factors = 1600, [1, 5, 1, 1, 10, 10]
        if epochval is not None: self.epochs = epochval

    def _init_adj_and_features(self):
        adj = adjacent_matrix_preprocessing(self.data['adata_omics1'], self.data['adata_omics2'], self.data['adj_emb'])
        self.adj_spatial_omics1 = adj['adj_spatial_omics1'].to(self.device)
        self.adj_spatial_omics2 = adj['adj_spatial_omics2'].to(self.device)
        self.adj_feature_omics1 = adj['adj_feature_omics1'].to(self.device)
        self.adj_feature_omics2 = adj['adj_feature_omics2'].to(self.device)
        self.adj_emb = adj['adj_emb'].to(self.device)
        self.features_omics1 = torch.FloatTensor(self.data['adata_omics1'].obsm['feat']).to(self.device)
        self.features_omics2 = torch.FloatTensor(self.data['adata_omics2'].obsm['feat']).to(self.device)
        self.dim_input1, self.dim_input2 = self.features_omics1.shape[1], self.features_omics2.shape[1]

    def train(self, epochs=None):
        epochs = epochs or self.epochs
        self.model = EncodingNetwork(self.dim_input1, self.dim_output, self.dim_input2, self.dim_output).to(self.device)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        for epoch in tqdm(range(epochs)):
            optimizer.zero_grad()
            f1, emb = (add_gaussian_noise(self.features_omics1, 0, 0.1), add_gaussian_noise(self.embedding, 0, 0.01)) if random.random() < 0.5 else (self.features_omics1, self.embedding)
            results = self.model(f1, self.features_omics2, self.adj_spatial_omics1, self.adj_feature_omics1, self.adj_spatial_omics2, self.adj_feature_omics2, emb, self.adj_emb)
            loss = (self.weight_factors[0]*F.mse_loss(self.features_omics1, results['emb_recon_omics1']) +
                    self.weight_factors[1]*F.mse_loss(self.features_omics2, results['emb_recon_omics2']) +
                    self.weight_factors[2]*F.mse_loss(results['emb_latent_omics1'], results['emb_cross1']) +
                    self.weight_factors[3]*F.mse_loss(results['emb_latent_omics2'], results['emb_cross2']) +
                    self.weight_factors[4]*F.mse_loss(self.embedding, results['emb_recon_spa']) +
                    self.weight_factors[5]*F.mse_loss(self.embedding, results['emb_recon_fea']))
            loss.backward()
            optimizer.step()
            self.loss_history.append(loss.item())
        return self._evaluate_model()

    def _evaluate_model(self):
        self.model.eval()
        with torch.no_grad():
            results = self.model(self.features_omics1, self.features_omics2, self.adj_spatial_omics1, self.adj_feature_omics1, self.adj_spatial_omics2, self.adj_feature_omics2, self.embedding, self.adj_emb)
        return {'emb_latent_omics1': F.normalize(results['emb_latent_omics1'], p=2).cpu().numpy(),
                'emb_latent_omics2': F.normalize(results['emb_latent_omics2'], p=2).cpu().numpy(),
                'SpatialGlue': F.normalize(results['emb_latent_combined'], p=2).cpu().numpy(),
                'alpha_omics1': results['alpha_omics1'].cpu().numpy(),
                'alpha_omics2': results['alpha_omics2'].cpu().numpy(),
                'alpha': results['alpha'].cpu().numpy()}

def mclust_R(adata, num_cluster, modelNames='EEE', used_obsm='emb_pca', random_seed=2020):
    import rpy2.robjects as robjects
    from rpy2.robjects import pandas2ri, default_converter
    from rpy2.robjects.conversion import localconverter
    np.random.seed(random_seed)
    robjects.r.library("mclust")
    robjects.r["set.seed"](random_seed)
    rmclust = robjects.r["Mclust"]
    X = np.array(adata.obsm[used_obsm], dtype=np.float64)
    df = pd.DataFrame(X, columns=[f'PC{i+1}' for i in range(X.shape[1])])
    subset_size = min(300, X.shape[0])
    subset_indices = robjects.IntVector(list(np.random.choice(range(1, X.shape[0] + 1), subset_size, replace=False)))
    init_list = robjects.ListVector({'subset': subset_indices})
    with localconverter(default_converter + pandas2ri.converter):
        res = rmclust(df, G=num_cluster, modelNames=modelNames, initialization=init_list)
    mclust_res = np.array(res['classification'])
    adata.obs['mclust'] = mclust_res
    adata.obs['mclust'] = adata.obs['mclust'].astype('int').astype('str').astype('category')
    return adata

def clustering(adata, n_clusters=7, key='emb', add_key='SpatialGlue', method='mclust', start=0.1, end=3.0, increment=0.01, use_pca=False, n_comps=20, random_seed=2020):
    if use_pca: adata.obsm[key + '_pca'] = pca(adata, use_reps=key, n_comps=n_comps)
    if method == 'mclust':
        used = key + '_pca' if use_pca else key
        adata = mclust_R(adata, used_obsm=used, num_cluster=n_clusters, random_seed=random_seed)
        adata.obs[add_key] = adata.obs['mclust']


# Automated Benchmark Harness across 6 Datasets and 20 Seeds (NicheFormer + spaLLM Flow)

In [ ]:
# Setup R_HOME and rpy2
os.environ['R_HOME'] = '/usr/lib/R'
os.environ['PATH'] = '/usr/lib/R/bin:' + os.environ['PATH']
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, default_converter
import rpy2.robjects.conversion as cv
cv.set_conversion(default_converter + pandas2ri.converter)
robjects.r.options(warn=-1)
robjects.r('''
if (!requireNamespace("mclust", quietly = TRUE)) {
    install.packages("mclust", repos="https://cloud.r-project.org")
}
library(mclust)
''')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Loading NicheFormer foundation model from HuggingFace...")
nicheformer_model = AutoModel.from_pretrained('theislab/Nicheformer', trust_remote_code=True).to(device)

choices = [
    ("10x_human_lymph_node_A1", "https://drive.google.com/drive/folders/10z1N4MwW8Y49o8GlkYGBKVx1N7fiMuyC"),
    ("10x_human_lymph_node_D1", "https://drive.google.com/drive/folders/1-g_Ca2XMaMXF-MisuVY-wobWDX86O6zz"),
    ("Mouse_Brain_E11_S1", "https://drive.google.com/drive/folders/1zRwDJrYnks0LRzlAVRqPU7jE_OcStgPo"),
    ("Mouse_Brain_E13_S1", "https://drive.google.com/drive/folders/1GOufwIRjjfcd9Bi2GKtebzKoPCg2jVud"),
    ("Mouse_Brain_E15_S1", "https://drive.google.com/drive/folders/1rHkTL5OF5qPsEERypRGMS51SjUQ69tdD"),
    ("Mouse_Brain_E18_S1", "https://drive.google.com/drive/folders/1Xj1LNIAY93biS6JIMKNRODn5GvtCKADB")
]

DATASET_INDICES = [0, 1, 2, 3, 4, 5]
SEEDS = [13, 2560, 641, 1892, 1173, 69, 2024, 231, 1971, 2497, 338, 3127, 2001, 2022, 574, 2428, 999, 1187, 42, 3999]
tool = 'mclust'

os.makedirs("results", exist_ok=True)
all_results = []

for dataset_idx in DATASET_INDICES:
    dataset_name, folder_url = choices[dataset_idx]
    print("\n" + "#"*80)
    print(f" STARTING DATASET: {dataset_name} ".center(80, "#"))
    print("#"*80)
    
    base = f"data/{dataset_name}"
    os.makedirs(base, exist_ok=True)
    rna_path = os.path.join(base, "adata_RNA.h5ad")
    
    if dataset_name.startswith("10x"):
        other_path = os.path.join(base, "adata_ADT.h5ad")
        annotation_path = os.path.join(base, "annotation.csv")
        gt_column = "manual-anno"
        data_type = '10x'
    else:
        other_path = os.path.join(base, "adata_ATAC.h5ad")
        annotation_path = os.path.join(base, "anno.csv")
        gt_column = "cluster"
        data_type = 'Spatial-epigenome-transcriptome'
        
    if not os.path.exists(rna_path) or not os.path.exists(other_path) or not os.path.exists(annotation_path):
        print(f"Downloading dataset files into: {base}")
        gdown_cmd = ".venv/bin/gdown" if os.path.exists(".venv/bin/gdown") else "gdown"
        os.system(f'{gdown_cmd} --folder "{folder_url}" --output "{base}"')
    else:
        print(f"Dataset files already exist at {base}. Skipping download.")
        
    adata_omics1 = sc.read_h5ad(rna_path)
    adata_omics2 = sc.read_h5ad(other_path)
    adata_omics1.var_names_make_unique()
    adata_omics2.var_names_make_unique()
    
    sc.pp.filter_genes(adata_omics1, min_cells=10)
    if not dataset_name.startswith("10x"):
        sc.pp.filter_cells(adata_omics1, min_genes=200)
        
    common_cells = adata_omics1.obs_names.intersection(adata_omics2.obs_names)
    adata_omics1 = adata_omics1[common_cells].copy()
    adata_omics2 = adata_omics2[common_cells].copy()
    
    anno_df = pd.read_csv(annotation_path, index_col=0)
    anno_df = anno_df.loc[anno_df.index.isin(common_cells)].copy()
    adata_omics1.obs['ground_truth'] = anno_df[gt_column]
    adata_omics2.obs['ground_truth'] = anno_df[gt_column]
    
    print(f"RNA cells: {adata_omics1.shape[0]} | Modality 2 cells: {adata_omics2.shape[0]}")
    
    # ---- NicheFormer Tokenization & Extraction ----
    print("Tokenizing RNA counts for NicheFormer...")
    rna_tokens = tokenize_for_nicheformer(adata_omics1.X, max_seq_len=1500)
    print("Extracting 512D embeddings via NicheFormer transformer...")
    emb_rna = extract_nicheformer_embeddings(nicheformer_model, rna_tokens, batch_size=16)
    adata_omics1.obsm['emb_rna'] = emb_rna.numpy()
    
    # Feature engineering for GNN reconstruction
    sc.pp.highly_variable_genes(adata_omics1, flavor="seurat_v3", n_top_genes=3000)
    sc.pp.normalize_total(adata_omics1, target_sum=1e4)
    sc.pp.log1p(adata_omics1)
    sc.pp.scale(adata_omics1)
    adata_omics1_high = adata_omics1[:, adata_omics1.var['highly_variable']]
    
    if dataset_name.startswith("10x"):
        adata_omics1.obsm["feat"] = pca(adata_omics1_high, n_comps=adata_omics2.n_vars - 1)
        adata_omics2 = clr_normalize_each_cell(adata_omics2)
        sc.pp.scale(adata_omics2)
        adata_omics2.obsm["feat"] = pca(adata_omics2, n_comps=adata_omics2.n_vars - 1)
    else:
        adata_omics1.obsm["feat"] = pca(adata_omics1_high, n_comps=50)
        adata_omics2 = adata_omics2[adata_omics1.obs_names].copy()
        if "X_lsi" not in adata_omics2.obsm:
            sc.pp.highly_variable_genes(adata_omics2, flavor="seurat_v3", n_top_genes=3000)
            lsi(adata_omics2, use_highly_variable=False, n_components=51)
        adata_omics2.obsm["feat"] = adata_omics2.obsm["X_lsi"].copy()
        
    data = construct_neighbor_graph(adata_omics1, adata_omics2, datatype=data_type)
    
    # Construct NicheFormer Embedding KNN Graph
    print("Building NicheFormer embedding KNN graph...")
    adj_emb = kneighbors_graph(adata_omics1.obsm['emb_rna'], 6, mode='connectivity', metric='correlation', include_self=False)
    data['adj_emb'] = adj_emb
    
    n_ground_truth = adata_omics1.obs["ground_truth"].nunique()
    print(f"Number of ground truth classes: {n_ground_truth}")
    
    dataset_results = []
    for seed in SEEDS:
        print(f"\n" + "-"*60)
        print(f" Dataset: {dataset_name} | Seed: {seed} ".center(60, "-"))
        print("-"*60)
        
        fix_seed(seed)
        data_copy = {
            'adata_omics1': data['adata_omics1'].copy(),
            'adata_omics2': data['adata_omics2'].copy(),
            'adj_emb': data['adj_emb'].copy()
        }
        
        model = Train_spaLLM(data_copy, embedding=emb_rna, datatype=data_type, device=device, random_seed=seed)
        output = model.train()
        
        adata = data_copy['adata_omics1'].copy()
        adata.obsm['SpatialGlue'] = output['SpatialGlue'].copy()
        clustering(adata, key='SpatialGlue', add_key='SpatialGlue', n_clusters=n_ground_truth, method=tool, use_pca=True, random_seed=seed)
        
        y_true = adata.obs['ground_truth'].astype(str)
        y_pred = adata.obs['SpatialGlue'].astype(str)
        ari = adjusted_rand_score(y_true, y_pred)
        nmi = normalized_mutual_info_score(y_true, y_pred)
        ami = adjusted_mutual_info_score(y_true, y_pred)
        homogeneity = homogeneity_score(y_true, y_pred)
        v_measure = v_measure_score(y_true, y_pred)
        
        joint_feat = adata.obsm['SpatialGlue']
        le = LabelEncoder()
        y_pred_int = le.fit_transform(y_pred)
        sil_score = silhouette_score(joint_feat, y_pred_int)
        
        print(f"Result -> ARI: {ari:.4f} | NMI: {nmi:.4f} | Silhouette: {sil_score:.4f}")
        res_dict = {'dataset': dataset_name, 'seed': seed, 'ARI': ari, 'NMI': nmi, 'AMI': ami,
                    'Homogeneity': homogeneity, 'V-measure': v_measure, 'Silhouette': sil_score}
        dataset_results.append(res_dict)
        all_results.append(res_dict)
        
    df_ds = pd.DataFrame(dataset_results)
    df_ds.to_csv(f"results/NicheQKV_v1_{dataset_name}_results.csv", index=False)
    print("\n" + "="*80)
    print(f" SUMMARY FOR {dataset_name} ".center(80, "="))
    print("="*80)
    print(df_ds.describe().loc[['mean', 'std']])
    print("="*80)

df_all = pd.DataFrame(all_results)
df_all.to_csv("results/NicheQKV_v1_all_results.csv", index=False)
print("\n" + "="*80)
print(" ALL 6 DATASETS COMPLETED SUCCESSFULLY ".center(80, "="))
print("="*80)
print(df_all.groupby('dataset')[['ARI', 'NMI', 'Silhouette']].mean())
print("="*80)
